In [1]:
from __future__ import print_function, division
%matplotlib inline
from matplotlib import pyplot as plt
import json
import random
import numpy as np

import debiaswe as dwe
from   debiaswe.we import WordEmbedding
from   debiaswe.data import load_professions
from   debiaswe.data import load_pairs

ImportError: cannot import name 'load_pairs' from 'debiaswe.data' (/Users/wjones/CC/MA120/MA120-FinalProject/debiaswe/data.py)

# What is hard-debiasing of vector word embeddings and how does it work mathmatically?

## 1. What are word-embeddings?

Word embeddings are how we turn natural language into vectors that computers can work with. There are a number of important word-embedding models, some more complicated than others. From a basic 1:1 model where every word corresponds to one vector, Word-2-Vec and GLoVE embeddings are the most common. These both work by iterating over large corpi of text, and finding context words for every word (that is, words that frequently are seen within proximity to the target word). By doing this for all words, a `m * n`  matrix is created, where `m` is the number of words in the vocabulary and `n` is the size of the embedding vector. 

Word embeddings capture semantic relationships between words, meaning that words with similar meanings are located close to each other in the vector space. This allows for various natural language processing tasks, such as sentiment analysis, machine translation, and information retrieval, to be performed more effectively. By representing words as dense vectors, word embeddings help in reducing the dimensionality of the data and preserving the contextual meaning of words.

Word2Vec is trained using a neural network with a single hidden layer. There are two main architectures for training Word2Vec: Continuous Bag of Words (CBOW) and Skip-gram. 

- **CBOW**: Predicts the target word (center word) from the context words (surrounding words). It maximizes the probability of the target word given the context words.
- **Skip-gram**: Predicts the context words from the target word. It maximizes the probability of the context words given the target word.

The training process involves the following steps:
1. Initialize the weights of the neural network randomly.
2. For each word in the corpus, create training samples based on the chosen architecture (CBOW or Skip-gram).
3. Use the training samples to update the weights of the neural network using backpropagation and gradient descent.
4. After training, the weights of the hidden layer are used as the word embeddings.

The objective function for training Word2Vec is to maximize the log probability of the context words given the target word (or vice versa), which can be computed using softmax.

## 2. What is the problem debiasing tries to solve?

Word embeddings are trained on huge amounts of natural language. For example, Word-2-Vec is commonly trained on all of Google News, or all of Wikipedia. Even more extreme, models like Bidirectional Encoder Representations from Transformers (BERT)
and GPT-models are trained on essentially the entire internet. As you may expect, this training data contains implict and explcit bias and prejudice against certain groups and people. This manifests itself clearly when we look at the problem of "man is to doctor  as woman is to nurse". We do this with vector aritmethic:  $\text{vector}(\text{woman}) + (\text{vector}(\text{doctor}) - \text{vector}(\text{man}))$ gives us nurse. You can see this below where we generate a basic vector for gender, and highlight which professions have a high connotation with gender. This is done by taking the gender vector and using cosine distance to determine which words from the given pairs are closest to it.

In [ ]:
# load google news word2vec
E = WordEmbedding('debiaswe/embeddings/w2v_gnews_small.txt')

# load professions
professions = load_professions()
profession_words = [p[0] for p in professions]

gender_vector_simple = E.v('she') - E.v('he') # compute a simple gender vector (just the difference between she and he)

a_gender = E.best_analogies_dist_thresh(gender_vector_simple)

for (a,b,c) in a_gender:
    print(a+"-"+b)

*** Reading data from debiaswe/embeddings/w2v_gnews_small.txt
(26423, 300)
26423 words of dimension 300 : in, for, that, is, ..., Jay, Leroy, Brad, Jermaine
Loaded professions
Format:
word,
definitional female -1.0 -> definitional male 1.0
stereotypical female -1.0 -> stereotypical male 1.0
Computing neighbors
Mean: 10.219732808538016
Median: 7.0
she-he
herself-himself
her-his
woman-man
daughter-son
businesswoman-businessman
girl-boy
actress-actor
chairwoman-chairman
heroine-hero
mother-father
spokeswoman-spokesman
sister-brother
girls-boys
sisters-brothers
queen-king
niece-nephew
councilwoman-councilman
motherhood-fatherhood
women-men
petite-lanky
ovarian_cancer-prostate_cancer
Anne-John
schoolgirl-schoolboy
granddaughter-grandson
aunt-uncle
matriarch-patriarch
twin_sister-twin_brother
mom-dad
lesbian-gay
husband-younger_brother
gal-dude
lady-gentleman
sorority-fraternity
mothers-fathers
grandmother-grandfather
blouse-shirt
soprano-baritone
queens-kings
Jill-Greg
daughters-sons
grandm

## 3. What does debiasing do?

## 4. How does debiasing work?

## 5. Example of debiasing

In [31]:

pairs = load_pairs()
pair_words = [p[0] for p in pairs]

difference_vectors = [E.diff(p[0], p[1]) for p in pairs]

D = np.vstack(difference_vectors)

U, S, Vt = np.linalg.svd(D, full_matrices=False)
v_bias = Vt[0]
v_bias /= np.linalg.norm(v_bias)

In [ ]:
# profession analysis gender
sp = sorted([(E.v(w).dot(v_bias), w) for w in pair_words])

sp[0:20], sp[-20:]

